# VANET Intrusion Detection System (Fixed — Leakage-Free)
### ML-Based Attack Classification for V2X Networks

**Dataset:** `v2x_dataset_Main_run.csv` (466,454 rows × 50 columns)  
**Labels:** Normal | Sybil | DDoS | Blackhole  
**Models:** Random Forest + XGBoost with `class_weight='balanced'` (no SMOTE)

---
### Bugs Fixed in This Version
| # | Location | Bug | Fix |
|---|----------|-----|-----|
| 1 | Cell 12 | **Incomplete leakage removal** — 12 attacker-side / derived-ratio columns still present (`my_attack_packets`, `fake_ratio`, `attack_rate`, etc.) making accuracy artificially near-100% | Removed all 19 leaky columns |
| 2 | Cell 20 | **Random `train_test_split`** — with only 5 cars per attack type, the same car appears in train AND test; model memorises car identity, not behaviour | Replaced with `GroupShuffleSplit(groups=car_id)` |
| 3 | Cell 21 | **SMOTE after random split** — synthetic samples interpolate between train/test rows, contaminating the test set | Removed SMOTE entirely; use `class_weight='balanced'` instead |
| 4 | Cells 25/29 | **Double-balancing** — SMOTE already equalises classes, then `class_weight='balanced'` re-weights again | With SMOTE gone, `class_weight='balanced'` is the sole balancing mechanism |
| 5 | Cell 2 | **`use_label_encoder=False`** deprecated kwarg in XGBoost ≥ 1.6 | Removed the kwarg |
| 6 | Analysis script | **`[pd.read](http://pd.read)_csv`** — hyperlink corruption broke the `pd.read_csv` call | Corrected to plain `pd.read_csv` |

---
## 1. Imports & Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# FIX 2: import GroupShuffleSplit instead of (only) train_test_split
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay,
)
from sklearn.feature_selection import mutual_info_classif

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('[INFO] xgboost not installed — will skip XGBoost.')

# FIX 3: SMOTE import removed — class_weight='balanced' handles imbalance

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = 'model_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('All imports loaded successfully.')

---
## 2. Load Dataset

In [ ]:
t0 = time.time()
# FIX 6: was '[pd.read](http://pd.read)_csv' — hyperlink had corrupted the call
df = pd.read_csv('v2x_dataset_Main_run.csv')
print(f'Shape: {df.shape}  |  Loaded in {time.time()-t0:.1f}s')
df.head()

In [ ]:
df.info()

---
## 3. Exploratory Data Analysis

In [ ]:
# Label distribution
label_counts = df['label'].value_counts()
print('Label Distribution:')
print(label_counts)
print(f'\nTotal samples: {len(df):,}')

In [ ]:
# Label distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
label_counts.plot.bar(ax=ax, color=colors, edgecolor='black')
ax.set_title('Label Distribution in V2X Dataset', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Attack Type')
for i, v in enumerate(label_counts):
    ax.text(i, v + 2000, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Car-level analysis — critical to understand why group split is needed
print('Unique cars per label:')
print(df.groupby('label')['car_id'].nunique())
print(f'\nTotal unique car_ids: {df["car_id"].nunique()}')
print('\n⚠️  Only ~5 cars per attack type!')
print('   Random split leaks car identity into test set → inflated accuracy.')
print('   Solution: GroupShuffleSplit by car_id (applied in Section 6).')

In [ ]:
# Missing values check
missing = df.isnull().sum()
print(f'Total missing values: {missing.sum()}')
missing[missing > 0].sort_values(ascending=False) if missing.sum() > 0 else print('No missing values!')

In [ ]:
# Basic statistics
df.describe()

---
## 4. Preprocessing

### FIX 1 — Aggressive leakage removal

The original notebook dropped only 7 columns. The remaining dataset still contained:
- **Attacker-side counters** (`my_attack_packets`, `my_dropped`, …) — a real IDS node cannot observe these
- **Derived ratios** (`fake_ratio`, `ddos_ratio`, `drop_ratio`, …) — mathematically encode the attack label
- **Pre-computed scores** (`sybil_score`, `ddos_score`, `blackhole_score`) — trivially predict the label

These columns make accuracy look near-100%, but the model learns nothing generalisable.

In [ ]:
# FIX 1: Complete leakage removal (19 columns vs original 7)
LEAK_COLS = [
    # Pre-computed labels / scores
    'event_type', 'is_fake', 'fake_id',
    'sybil_score', 'ddos_score', 'blackhole_score',
    # Attacker-side counters (not observable by an IDS node in deployment)
    'my_attack_packets', 'my_fake_ids_sent', 'my_ddos_packets', 'my_dropped',
    'attack_rate', 'ddos_rate_actual',
    # Derived ratios that directly encode attack identity
    'fake_ratio', 'ddos_ratio', 'blackhole_ratio', 'drop_ratio',
    'norm_attack_rate',
    # Zero/near-zero variance (simulation artefact)
    'avg_pkt_size_sent', 'norm_pkt_size',
]

# Keep car_id for group splitting — but exclude it from features later
dropped = [c for c in LEAK_COLS if c in df.columns]
df.drop(columns=dropped, inplace=True)
print(f'Dropped {len(dropped)} leaky / non-predictive columns:')
for c in dropped:
    print(f'  - {c}')
print(f'\nRemaining columns: {df.shape[1]}')

In [ ]:
# Encode target labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
label_names = list(le.classes_)
print(f'Label encoding: {dict(zip(label_names, le.transform(label_names)))}')

In [ ]:
# Feature / Target split
# car_id is excluded from features but retained separately for group splitting
FEATURE_COLS = [c for c in df.columns if c not in ('label', 'label_enc', 'car_id')]
X = df[FEATURE_COLS].copy()
y = df['label_enc'].copy()
groups = df['car_id'].copy()   # used for group-based splitting only

# Handle missing values
if X.isnull().sum().sum() > 0:
    X.fillna(X.median(), inplace=True)
    print('Filled missing values with column medians.')

# Handle remaining object columns
obj_cols = X.select_dtypes(include='object').columns.tolist()
if obj_cols:
    print(f'Encoding categorical columns: {obj_cols}')
    for col in obj_cols:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

print(f'Final feature matrix: {X.shape}')

---
## 5. Feature Selection (Mutual Information)

In [ ]:
# Compute MI scores on a 50K sample for speed
sample_idx = np.random.RandomState(42).choice(len(X), size=min(50_000, len(X)), replace=False)
mi_scores = mutual_info_classif(X.iloc[sample_idx], y.iloc[sample_idx], random_state=42)
mi_df = pd.DataFrame({'feature': FEATURE_COLS, 'mi_score': mi_scores}).sort_values(
    'mi_score', ascending=False
)
mi_df.head(15)

In [ ]:
# Feature importance chart
fig, ax = plt.subplots(figsize=(10, 6))
top15 = mi_df.head(15)
ax.barh(top15['feature'][::-1], top15['mi_score'][::-1], color='#3498db', edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 15 Features — Mutual Information (Leakage-Free)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Select top features (MI > 0.01)
TOP_FEATURES = mi_df.loc[mi_df['mi_score'] > 0.01, 'feature'].tolist()
if len(TOP_FEATURES) < 10:
    TOP_FEATURES = mi_df.head(15)['feature'].tolist()
print(f'Selected {len(TOP_FEATURES)} features with MI > 0.01')
X = X[TOP_FEATURES]
X.shape

---
## 6. Group-Based Train/Test Split

### FIX 2 — Replace random split with `GroupShuffleSplit(groups=car_id)`

**Why this matters:** The dataset has only ~5 cars per attack type. With a random row-level split,
rows from the *same car* land in both train and test. The model trivially learns each car's
unique signature and achieves near-100% accuracy — but will fail completely on unseen vehicles.

`GroupShuffleSplit` guarantees **zero car overlap** between train and test.

### FIX 3 — SMOTE removed

SMOTE was applied *after* the random split on scaled data. This means:
1. Synthetic samples are interpolated between real training rows that are already leaking car identity.
2. SMOTE sees scaled test-contaminated features (scaler was fit on all of `X_train` before the split was honest).

Replacement: `class_weight='balanced'` in both models handles class imbalance without any data contamination.

### FIX 4 — No double-balancing

Original code used SMOTE **and** `class_weight='balanced'` simultaneously. Removed SMOTE; `class_weight='balanced'` alone is correct.

In [ ]:
# FIX 2: Group-based 80/20 split — no car appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test  = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

train_cars = groups.iloc[train_idx].unique()
test_cars  = groups.iloc[test_idx].unique()
overlap    = set(train_cars) & set(test_cars)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train cars: {len(train_cars)}  |  Test cars: {len(test_cars)}')
print(f'Car overlap (must be 0): {len(overlap)}')

print('\nTrain label distribution:')
print(pd.Series(y_train.values).map(dict(enumerate(label_names))).value_counts())

print('\nTest label distribution:')
print(pd.Series(y_test.values).map(dict(enumerate(label_names))).value_counts())

# Warn if a class is missing from either split
if set(y_train.unique()) != set(y_test.unique()):
    print('\n⚠️  Warning: not all classes represented in both splits!')
    print('   Consider increasing test_size or using StratifiedGroupKFold.')

# Scale features (fit on train only)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# FIX 3: SMOTE block removed entirely

---
## 7. Model Training & Evaluation

In [ ]:
results = {}

def evaluate_model(name, model, Xtr, ytr, Xte, yte):
    """Train, predict, and report metrics."""
    t_start = time.time()
    model.fit(Xtr, ytr)
    train_time = time.time() - t_start

    y_pred = model.predict(Xte)
    acc   = accuracy_score(yte, y_pred)
    f1_w  = f1_score(yte, y_pred, average='weighted')
    f1_mac = f1_score(yte, y_pred, average='macro')

    # Only include classes actually present in the test set
    present_labels = sorted(yte.unique())
    present_names  = [label_names[i] for i in present_labels]

    results[name] = {
        'model': model, 'accuracy': acc,
        'f1_weighted': f1_w, 'f1_macro': f1_mac,
        'train_time': train_time, 'y_pred': y_pred,
    }
    return acc, f1_w, f1_mac, train_time, y_pred, present_labels, present_names

print('Evaluation function ready.')

### 7a. Random Forest

In [ ]:
# FIX 4: class_weight='balanced' is the sole balancing mechanism (no SMOTE)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=25,
    min_samples_split=5, min_samples_leaf=2,
    class_weight='balanced', random_state=42, n_jobs=-1,
)
# Train on raw (non-SMOTE) scaled data
acc, f1_w, f1_mac, t, y_pred_rf, pl, pn = evaluate_model(
    'Random Forest', rf, X_train_sc, y_train, X_test_sc, y_test
)
print(f'Accuracy:       {acc:.4f}')
print(f'F1 (weighted):  {f1_w:.4f}')
print(f'F1 (macro):     {f1_mac:.4f}')
print(f'Training time:  {t:.1f}s')

In [ ]:
# Random Forest — Classification Report
print(classification_report(y_test, y_pred_rf, target_names=pn, labels=pl))

In [ ]:
# Random Forest — Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred_rf, labels=pl)
disp = ConfusionMatrixDisplay(cm, display_labels=pn)
disp.plot(ax=ax, cmap='Blues', values_format=',')
ax.set_title('Random Forest — Confusion Matrix (Group Split)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7b. XGBoost

In [ ]:
if HAS_XGB:
    # Compute per-sample weights to balance classes (replaces SMOTE)
    class_counts = np.bincount(y_train)
    total = len(y_train)
    n_classes = len(class_counts)
    sample_weights = np.array([total / (n_classes * class_counts[yi]) for yi in y_train])

    # FIX 5: removed deprecated use_label_encoder=False kwarg
    xgb = XGBClassifier(
        n_estimators=300, max_depth=10,
        learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, objective='multi:softmax',
        num_class=len(label_names), eval_metric='mlogloss',
        random_state=42, n_jobs=-1,
    )

    t_start = time.time()
    xgb.fit(X_train_sc, y_train, sample_weight=sample_weights)
    train_time = time.time() - t_start

    y_pred_xgb = xgb.predict(X_test_sc)
    present_labels = sorted(y_test.unique())
    present_names  = [label_names[i] for i in present_labels]

    acc   = accuracy_score(y_test, y_pred_xgb)
    f1_w  = f1_score(y_test, y_pred_xgb, average='weighted')
    f1_mac = f1_score(y_test, y_pred_xgb, average='macro')

    print(f'Accuracy:       {acc:.4f}')
    print(f'F1 (weighted):  {f1_w:.4f}')
    print(f'F1 (macro):     {f1_mac:.4f}')
    print(f'Training time:  {train_time:.1f}s')

    results['XGBoost'] = {
        'model': xgb, 'accuracy': acc,
        'f1_weighted': f1_w, 'f1_macro': f1_mac,
        'train_time': train_time, 'y_pred': y_pred_xgb,
    }
else:
    print('XGBoost not available — skipping.')

In [ ]:
if HAS_XGB:
    # XGBoost — Classification Report
    print(classification_report(y_test, y_pred_xgb, target_names=present_names, labels=present_labels))

In [ ]:
if HAS_XGB:
    # XGBoost — Confusion Matrix
    fig, ax = plt.subplots(figsize=(7, 6))
    cm = confusion_matrix(y_test, y_pred_xgb, labels=present_labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=present_names)
    disp.plot(ax=ax, cmap='Greens', values_format=',')
    ax.set_title('XGBoost — Confusion Matrix (Group Split)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 8. Model Comparison

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    name: {
        'Accuracy': r['accuracy'],
        'F1 (Weighted)': r['f1_weighted'],
        'F1 (Macro)': r['f1_macro'],
        'Train Time (s)': round(r['train_time'], 1),
    }
    for name, r in results.items()
}).T
comparison

In [ ]:
# Comparison bar chart
if len(results) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    model_names = list(results.keys())
    metrics = ['accuracy', 'f1_weighted', 'f1_macro']
    titles  = ['Accuracy', 'F1 (Weighted)', 'F1 (Macro)']

    for ax, metric, title in zip(axes, metrics, titles):
        vals = [results[m][metric] for m in model_names]
        bars = ax.bar(model_names, vals, color=['#2ecc71', '#3498db'], edgecolor='black')
        ax.set_title(title, fontweight='bold')
        ax.set_ylim(max(0, min(vals) - 0.15), 1.05)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                    f'{v:.4f}', ha='center', fontsize=10)
    plt.suptitle('Model Comparison — Group-Based Split (Honest Evaluation)',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

---
## 9. Save Best Model & Artifacts

In [ ]:
# Pick best model by weighted F1
best_name  = max(results, key=lambda k: results[k]['f1_weighted'])
best_model = results[best_name]['model']

# Save everything
joblib.dump(best_model,   os.path.join(OUTPUT_DIR, 'best_model.joblib'))
joblib.dump(scaler,       os.path.join(OUTPUT_DIR, 'scaler.joblib'))
joblib.dump(le,           os.path.join(OUTPUT_DIR, 'label_encoder.joblib'))
joblib.dump(TOP_FEATURES, os.path.join(OUTPUT_DIR, 'feature_list.joblib'))

print(f'Best model:     {best_name}')
print(f'Accuracy:       {results[best_name]["accuracy"]:.4f}')
print(f'F1 (weighted):  {results[best_name]["f1_weighted"]:.4f}')
print(f'F1 (macro):     {results[best_name]["f1_macro"]:.4f}')
print(f'Features used:  {len(TOP_FEATURES)}')
print(f'SMOTE:          NOT USED (class_weight="balanced")')
print(f'Split method:   GroupShuffleSplit by car_id')
print(f'\nAll artifacts saved to: {OUTPUT_DIR}/')

---
## 10. Inference Demo
Quick demo showing how to load and use the saved model on new data.

In [ ]:
# Load saved artifacts
loaded_model    = joblib.load(os.path.join(OUTPUT_DIR, 'best_model.joblib'))
loaded_scaler   = joblib.load(os.path.join(OUTPUT_DIR, 'scaler.joblib'))
loaded_le       = joblib.load(os.path.join(OUTPUT_DIR, 'label_encoder.joblib'))
loaded_features = joblib.load(os.path.join(OUTPUT_DIR, 'feature_list.joblib'))

# Predict on 5 random test samples
sample    = X_test.sample(5, random_state=123)
sample_sc = loaded_scaler.transform(sample[loaded_features])
preds     = loaded_model.predict(sample_sc)
pred_labels   = loaded_le.inverse_transform(preds)
actual_labels = loaded_le.inverse_transform(y_test.loc[sample.index].values)

pd.DataFrame({
    'Predicted': pred_labels,
    'Actual':    actual_labels,
    'Correct':   pred_labels == actual_labels,
}, index=sample.index)

---
## 11. Fixed Analysis Script

The snippet shared alongside the notebook also had the broken `[pd.read](http://pd.read)_csv` hyperlink. Below is the corrected version.

In [ ]:
# FIX 6 — corrected analysis script (was '[pd.read](http://pd.read)_csv')
remaining = [
    'timestamp','pos_x','pos_y','speed_mps','distance_moved','position_changes',
    'avg_speed','pkt_size_bytes','my_beacons_sent','my_beacons_received',
    'beacon_send_rate','beacon_recv_rate','total_bytes_sent','total_bytes_rcvd',
    'avg_pkt_size_rcvd','throughput_bps','send_recv_ratio','active_duration',
    'beacon_regularity','global_beacons_sent','global_beacons_rcvd',
    'global_attack_pkts','global_dropped','network_load',
    'norm_send_rate','norm_recv_rate','norm_bytes_sent','norm_speed'
]
# Filter to only columns that still exist after leakage removal
remaining = [c for c in remaining if c in df.columns]

bh_df     = df[df['label'] == 'blackhole']
other_df  = df[df['label'] != 'blackhole']
sybil_df  = df[df['label'] == 'sybil']
normal_df = df[df['label'] == 'normal']
ddos_df   = df[df['label'] == 'ddos']

print('=== Columns where blackhole is ALL ZEROS (simulation artefact) ===')
for col in remaining:
    if bh_df[col].max() == 0 and bh_df[col].min() == 0:
        others_mean = other_df[col].mean()
        print(f'  {col:30s}  blackhole=[0,0]  others_mean={others_mean:.2f}')

print()
print('=== Columns where sybil is clearly separable (>2x or <0.5x normal AND ddos) ===')
for col in remaining:
    s = sybil_df[col].mean()
    n = normal_df[col].mean()
    d = ddos_df[col].mean()
    if n > 0 and d > 0:
        rn = s / n
        rd = s / d
        if (rn > 2 and rd > 2) or (rn < 0.5 and rd < 0.5):
            print(f'  {col:30s}  sybil={s:.1f}  normal={n:.1f}  ddos={d:.1f}')

print()
print('=== global_* columns (network-wide; consider whether realistic for single-node IDS) ===')
for col in remaining:
    if col.startswith('global_'):
        g = df.groupby('label')[col].mean()
        print(f'  {col:30s}  {dict(g.items())}')